# NILM-Engine — Colab GCS 학습 (EXP1~EXP4)

| 항목 | 경로 |
|------|------|
| 원천데이터 | `nilm/training_dev10/household_id=.../channel=.../date=.../` |
| 라벨데이터 | `nilm/labels/training.parquet` (5.9MB) |
| Train | house_011, 015, 016, 017, 033, 039, 054, 063 |
| Val | house_049 |
| Test | house_067 |

**실행 순서**: 1→2→3→4→5(선택) → **6~9 (학습)**

## 1. 환경 설치 & GCS 인증

In [ ]:
!pip install -q pyarrow torch

In [ ]:
from google.colab import auth
auth.authenticate_user()
print("GCS 인증 완료")

## 2. GCS 연결 검증

In [ ]:
!gcloud storage ls gs://ax-nilm-data-dhwang0803-us/nilm/training_dev10/ | head -5

In [ ]:
import gcsfs
import pyarrow.dataset as ds
import pyarrow.parquet as pq
from pyarrow.fs import PyFileSystem, FSSpecHandler

gcs = gcsfs.GCSFileSystem()
_gcs_pa = PyFileSystem(FSSpecHandler(gcs))  # pyarrow 직접 읽기용

print("=== 원천데이터 스키마 ===")
raw_ds = ds.dataset(
    "ax-nilm-data-dhwang0803-us/nilm/training_dev10",
    filesystem=_gcs_pa, partitioning=["household_id", "channel", "date"],
)
print(raw_ds.schema)

print("\n=== 라벨 스키마 ===")
label_tbl = pq.read_table(
    "ax-nilm-data-dhwang0803-us/nilm/labels/training.parquet", filesystem=_gcs_pa
)
print(label_tbl.schema)
print(f"\n행수: {label_tbl.num_rows:,}")

In [ ]:
# 1일치 행수 — 2,592,000 (30Hz × 86400s) 이면 OK
t = pq.read_table(
    "ax-nilm-data-dhwang0803-us/nilm/training_dev10/household_id=house_011/channel=ch01/date=20231004/part-0.parquet",
    filesystem=_gcs_pa,
)
print(f"행수: {t.num_rows:,}")

## 3. 분할 & 경로 상수

In [ ]:
SPLIT = {
    "train": ["house_011", "house_015", "house_016", "house_017",
              "house_033", "house_039", "house_054", "house_063"],
    "val":   ["house_049"],
    "test":  ["house_067"],
}
BUCKET_PREFIX = "ax-nilm-data-dhwang0803-us/nilm/training_dev10"
LABEL_PATH    = "ax-nilm-data-dhwang0803-us/nilm/labels/training.parquet"
MODELS        = ["seq2point"] # 여기서 보인 모델만 선택해서 진행  ["seq2point", "bert4nilm", "cnn_tda"]

print(f"Train {len(SPLIT['train'])}개 / Val {len(SPLIT['val'])}개 / Test {len(SPLIT['test'])}개")

## 4. 리포지토리 설정

In [ ]:
import os, sys

REPO_DIR = "/content/ax_nilm"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/dhwang0803-glitch/ax_nilm {REPO_DIR} -q
    print("클론 완료")
else:
    !git -C {REPO_DIR} pull -q && echo "최신화 완료"

SRC_DIR     = f"{REPO_DIR}/nilm-engine/src"
SCRIPTS_DIR = f"{REPO_DIR}/nilm-engine/scripts"
CFG_DIR     = f"{REPO_DIR}/nilm-engine/config"

for d in [SRC_DIR, SCRIPTS_DIR]:
    if d not in sys.path:
        sys.path.insert(0, d)
print("경로 설정 완료")

In [ ]:
import yaml

from acquisition.gcs_loader import (
    list_channels_gcs, get_house_start_date_gcs,
    load_channel_data_gcs, get_appliance_name_gcs,
    GCSNILMDataset,
)
from train_model import (
    build_model, masked_mse, evaluate, train_one_epoch, _NILMDatasetWithTDA
)

with open(f"{CFG_DIR}/train.yaml")   as f: TRAIN_CFG   = yaml.safe_load(f)
with open(f"{CFG_DIR}/dataset.yaml") as f: DATASET_CFG = yaml.safe_load(f)

print("모든 모듈 import 완료")

## 5. 데이터 탐색 (선택)

In [ ]:
print(f"{'house':12s} {'채널수':>6s}  {'시작일':>12s}")
print("-" * 36)
for house_id in SPLIT["train"] + SPLIT["val"] + SPLIT["test"]:
    channels = list_channels_gcs(gcs, house_id, BUCKET_PREFIX)
    try:
        start = get_house_start_date_gcs(gcs, house_id, bucket_prefix=BUCKET_PREFIX)
    except FileNotFoundError:
        start = "N/A"
    print(f"{house_id:12s} {len(channels):>6d}  {str(start):>12s}")

In [ ]:
# house_011 채널→가전 매핑
for ch in list_channels_gcs(gcs, "house_011", BUCKET_PREFIX):
    print(f"  {ch}: {get_appliance_name_gcs(gcs, 'house_011', ch, LABEL_PATH)}")

## 6. Google Drive 마운트 (체크포인트 영구 저장)

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

CKPT_DIR    = Path("/content/drive/MyDrive/nilm/checkpoints")
RESULTS_DIR = Path("/content/drive/MyDrive/nilm/results")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"체크포인트 → {CKPT_DIR}")
print(f"결과       → {RESULTS_DIR}")

## 7. EXP 실행 함수 정의

In [ ]:
import json, time
import torch
from torch.utils.data import DataLoader


def run_exp_gcs(exp_name: str, model_name: str) -> dict:
    """GCS 데이터로 단일 EXP/모델 학습. 결과를 Drive에 저장하고 metrics dict 반환."""
    exp_cfg = TRAIN_CFG["experiments"][exp_name]
    week = exp_cfg.get("week")
    ws   = DATASET_CFG["window"]["size"]
    st   = DATASET_CFG["window"]["stride"]
    bs   = TRAIN_CFG["training"]["batch_size"]
    ep   = TRAIN_CFG["training"]["epochs"]
    pat  = TRAIN_CFG["training"]["early_stopping_patience"]
    lr   = TRAIN_CFG["training"]["learning_rate"]
    wd   = TRAIN_CFG["optimizer"]["weight_decay"]

    print(f"\n{'='*58}")
    print(f"  {exp_name} / {model_name}  (train=week {week}, val=weeks 1~{week})")
    print(f"{'='*58}")

    # ── 1. 데이터셋 ──────────────────────────────────────────────────────────
    print("  [1/4] 데이터셋 로딩 (GCS → 메모리)...")
    base_train = GCSNILMDataset(
        SPLIT["train"], gcs_fs=gcs,
        bucket_prefix=BUCKET_PREFIX, label_path=LABEL_PATH,
        window_size=ws, stride=st, week=week, fit_scaler=True,
    )
    base_val = GCSNILMDataset(
        SPLIT["val"], gcs_fs=gcs,
        bucket_prefix=BUCKET_PREFIX, label_path=LABEL_PATH,
        window_size=ws, stride=st,
        max_week=week,             # weeks 1..week 누적 (학습 누적량과 동일)
        scaler=base_train.scaler,  # train 기준 정규화
    )
    if model_name == "cnn_tda":
        train_ds = _NILMDatasetWithTDA(base_train)
        val_ds   = _NILMDatasetWithTDA(base_val)
    else:
        train_ds, val_ds = base_train, base_val
    train_dl = DataLoader(train_ds, batch_size=bs, shuffle=True,  num_workers=2, pin_memory=True)
    val_dl   = DataLoader(val_ds,   batch_size=bs, shuffle=False, num_workers=2, pin_memory=True)
    print(f"  train={len(train_ds):,}  val={len(val_ds):,} windows")

    # ── 2. 모델 ──────────────────────────────────────────────────────────────
    print("  [2/4] 모델 초기화...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"  device: {device}")
    model = build_model(model_name, ws).to(device)

    resume_exp = exp_cfg.get("resume_from")
    if resume_exp:
        prev_ckpt = CKPT_DIR / f"{resume_exp}_{model_name}.pt"
        if prev_ckpt.exists():
            model.load_state_dict(torch.load(prev_ckpt, map_location=device))
            print(f"  └─ 체크포인트 로드: {prev_ckpt.name}")
        else:
            print(f"  └─ 경고: {prev_ckpt.name} 없음 — 처음부터 학습")

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, "min",
        factor=TRAIN_CFG["scheduler"]["factor"],
        patience=TRAIN_CFG["scheduler"]["patience"],
    )

    # ── 3. 학습 루프 ─────────────────────────────────────────────────────────
    print("  [3/4] 학습 시작...")
    best_mae, best_state, no_improve = float("inf"), None, 0
    t0 = time.perf_counter()

    for epoch in range(1, ep + 1):
        loss = train_one_epoch(model, train_dl, optimizer, model_name, device)
        vm   = evaluate(model, val_dl, model_name, device)
        scheduler.step(vm["mae"])
        lr_now = optimizer.param_groups[0]["lr"]
        print(f"  ep {epoch:3d}/{ep}  loss={loss:.4f}  "
              f"val_mae={vm['mae']:.2f}W  r2={vm['r2']:.4f}  lr={lr_now:.2e}")

        if vm["mae"] < best_mae - 1e-4:
            best_mae = vm["mae"]
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= pat:
                print(f"  조기 종료 ({pat} epoch 개선 없음)")
                break

    elapsed = time.perf_counter() - t0

    # ── 4. 저장 ──────────────────────────────────────────────────────────────
    print("  [4/4] 체크포인트 & 메트릭 저장...")
    if best_state:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

    torch.save(model.state_dict(), CKPT_DIR / f"{exp_name}_{model_name}.pt")

    fm = evaluate(model, val_dl, model_name, device)
    fm.update({"exp": exp_name, "model": model_name, "week": week,
               "val_weeks": f"1~{week}", "training_time_s": round(elapsed, 1)})
    with open(RESULTS_DIR / f"{exp_name}_{model_name}_metrics.json", "w") as f:
        json.dump(fm, f, ensure_ascii=False, indent=2)

    print(f"  ✅  MAE={fm['mae']:.2f}W  RMSE={fm['rmse']:.2f}W  "
          f"SAE={fm['sae']:.4f}  F1={fm['f1']:.3f}  ({elapsed:.0f}s)")
    return fm


def write_results_md(exp_name: str) -> None:
    """EXP 결과 MD를 RESULTS_DIR에 생성 (Drive에 저장됨)."""
    from datetime import datetime

    exp_cfg  = TRAIN_CFG["experiments"][exp_name]
    week     = exp_cfg.get("week", "?")
    prev_exp = exp_cfg.get("resume_from")

    exp_ms  = {}
    prev_ms = {}
    for m in MODELS:
        p = RESULTS_DIR / f"{exp_name}_{m}_metrics.json"
        exp_ms[m] = json.load(open(p)) if p.exists() else None
        pp = RESULTS_DIR / f"{prev_exp}_{m}_metrics.json"
        prev_ms[m] = (json.load(open(pp)) if pp.exists() else None) if prev_exp else None

    lines = [
        f"# {exp_name} 실험 결과", "",
        "| 항목 | 값 |", "|------|-----|",
        f"| 학습 주차 | week {week} (house별 시작일 기준 {(week-1)*7+1 if isinstance(week,int) else '?'}~{week*7 if isinstance(week,int) else '?'}일차) |",
        f"| 이전 체크포인트 | {prev_exp or '처음부터'} |",
        f"| 기록 일시 | {datetime.now().strftime('%Y-%m-%d %H:%M')} |",
        "", "---", "",
        "## 전체 성능 비교", "",
        "| 모델 | MAE (W) | RMSE (W) | SAE | F1 |",
        "|------|---------|----------|-----|----|",
    ]
    for m in MODELS:
        md = exp_ms.get(m)
        if md:
            lines.append(f"| {m} | {md['mae']:.2f} | {md['rmse']:.2f} | {md['sae']:.4f} | {md['f1']:.3f} |")
        else:
            lines.append(f"| {m} | — | — | — | — |")

    lines += ["", "---", "", "## 이전 EXP 대비 개선율 (Val MAE)"]
    if not prev_exp:
        lines += ["", "> 첫 번째 실험 — 비교 대상 없음"]
    else:
        lines += ["", "| 모델 | 이전 MAE | 현재 MAE | 개선율 |",
                  "|------|---------|---------|--------|",]
        for m in MODELS:
            cur, prv = exp_ms.get(m), prev_ms.get(m)
            if cur and prv:
                imp = (prv["mae"] - cur["mae"]) / (prv["mae"] + 1e-8) * 100
                trend = "↓" if imp > 0 else "↑"
                lines.append(f"| {m} | {prv['mae']:.2f}W | {cur['mae']:.2f}W | {trend} {abs(imp):.1f}% |")
            else:
                lines.append(f"| {m} | — | — | — |")

    lines += ["", "---", "", "## 메모", "",
              "> 특이사항, 하이퍼파라미터 변경 내역 등을 여기에 기록하세요.", ""]

    md_path = RESULTS_DIR / f"{exp_name}_results.md"
    md_path.write_text("\n".join(lines), encoding="utf-8")
    print(f"MD 생성: {md_path}")


print("함수 정의 완료 — run_exp_gcs / write_results_md")

## 8. EXP1 실행

> ⚠️ **메모리**: 8 houses × 7일 × 30Hz ≈ 1.4억 샘플.  
> **시간**: GPU 기준 seq2point ~20min, bert4nilm ~30min, cnn_tda ~60min (epoch당).

In [ ]:
results = {}

for model_name in MODELS:
    results[("EXP1", model_name)] = run_exp_gcs("EXP1", model_name)

write_results_md("EXP1")

## 9. EXP2~4 연속 실행 (포화점 자동 감지)

이전 EXP 대비 Val MAE 평균 개선율 < `saturation_threshold` (5%) 이면 자동 중단.

In [ ]:
THR = TRAIN_CFG["saturation_threshold"] * 100  # 5.0 (%)

for exp_name in ["EXP2", "EXP3", "EXP4"]:
    for model_name in MODELS:
        results[(exp_name, model_name)] = run_exp_gcs(exp_name, model_name)
    write_results_md(exp_name)

    # 포화점 판단
    prev_exp = TRAIN_CFG["experiments"][exp_name]["resume_from"]
    imps = []
    for m in MODELS:
        cur = results.get((exp_name, m), {}).get("mae")
        prv_path = RESULTS_DIR / f"{prev_exp}_{m}_metrics.json"
        if cur and prv_path.exists():
            prv_mae = json.load(open(prv_path))["mae"]
            imps.append((prv_mae - cur) / (prv_mae + 1e-8) * 100)

    if imps:
        avg = sum(imps) / len(imps)
        if avg < THR:
            print(f"\n⚠️  {exp_name}: 평균 개선 {avg:.1f}% < 기준 {THR:.0f}% → 포화 — 학습 중단")
            break
        else:
            print(f"\n✅  {exp_name}: 평균 개선 {avg:.1f}% → 다음 EXP 진행")

## 10. 결과 요약 조회

In [ ]:
print(f"{'EXP':6s} {'모델':14s} {'MAE':>8s} {'RMSE':>8s} {'SAE':>8s} {'F1':>6s}")
print("-" * 56)
for (exp_name, model_name), m in sorted(results.items()):
    if m:
        print(f"{exp_name:6s} {model_name:14s} {m['mae']:8.2f} {m['rmse']:8.2f} {m['sae']:8.4f} {m['f1']:6.3f}")

print(f"\nMD 파일 위치: {RESULTS_DIR}/EXP*_results.md")